# Demo 03 · Agente comercial con trazabilidad y evaluación

            Esta demo simula un agente comercial de FraSoHome que combina cuatro herramientas: modelo de inactividad, explicación local, RAG documental y CRM. El objetivo es enseñar cómo se audita un agente por pasos, no solo por la respuesta final.


In [1]:
from pathlib import Path
import sys
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datos").exists() and (candidate / "escenarios").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado la raíz del proyecto FraSoHome.")

PROJECT_ROOT = find_project_root()
DEMO_DIR = PROJECT_ROOT / "escenarios" / "09-genai-interpretabilidad"
ARTIFACT_DIR = DEMO_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DEMO_DIR))

from common.genai_frasohome import (
    append_trace,
    build_agent_trace,
    build_rag_index,
    chunk_documents,
    evaluate_agent_trace,
    evaluate_rag_questions,
    grounded_answer,
    keyword_coverage,
    load_crm,
    load_knowledge_documents,
    load_sales_artifact,
    local_feature_contributions,
    recommendation_from_score,
    retrieve,
    simple_answer_rubric,
    summarize_customer_context,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEMO_DIR:", DEMO_DIR)


PROJECT_ROOT: C:\repos\frasohome-demos
DEMO_DIR: C:\repos\frasohome-demos\escenarios\09-genai-interpretabilidad


## 1. Preparar herramientas del agente


In [2]:
artifact = load_sales_artifact(PROJECT_ROOT)
churn = artifact["churn"]
crm = load_crm(PROJECT_ROOT)

documents = load_knowledge_documents(PROJECT_ROOT)
chunks = chunk_documents(documents)
rag_index = build_rag_index(chunks, include_obsolete=False, include_security_test=False)

print("Artefacto ventas:", artifact["_path"])
print("Clientes CRM:", crm["customer_id_std"].nunique())
print("Chunks RAG:", len(rag_index.chunks))


Artefacto ventas: C:\repos\frasohome-demos\escenarios\04-analitica-ventas-ml\artifacts\sales_models_artifact.joblib
Clientes CRM: 100
Chunks RAG: 33


## 2. Definir la solicitud de negocio

            El agente recibe una tarea ambigua pero realista. Debe entenderla, planificar, consultar herramientas, respetar guardrails y responder con evidencia.


In [3]:
user_request = "Selecciona clientes Platino con alto riesgo de inactividad y prepara una propuesta de campaña respetando consentimiento, margen y presión comercial."
trace = build_agent_trace(user_request)
append_trace(trace, "intent", "reactivación comercial de clientes de alto valor con restricciones de gobierno")
append_trace(trace, "plan", "1) obtener cliente con score alto; 2) explicar drivers; 3) consultar reglas CRM; 4) comprobar consentimiento; 5) generar recomendación con cautelas")
display(pd.DataFrame(trace))


,step,type,detail,payload
0,0,user_request,"Selecciona clientes Platino con alto riesgo de inactividad y prepara una propuesta de campaña respetando consentimiento, margen y presión comercial.",NaN
1,1,intent,reactivación comercial de clientes de alto valor con restricciones de gobierno,NaN
2,2,plan,1) obtener cliente con score alto; 2) explicar drivers; 3) consultar reglas CRM; 4) comprobar consentimiento; 5) generar recomendación con cautelas,NaN


## 3. Tool 1: modelo de inactividad + explicación local


In [4]:
append_trace(trace, "tool_call", "churn_model.predict_proba + explicación local", {"target": "label_inactive_90d"})
score, row_raw, contributions = local_feature_contributions(churn["result"], top_n=10)
customer_context = summarize_customer_context(row_raw, contributions)
append_trace(trace, "tool_result", "cliente prioritario y señales principales", {"score": round(score, 4), "signals": contributions["feature"].head(5).tolist()})

print(f"Score seleccionado: {score:.3f}")
display(row_raw.T.head(25))
display(contributions)


Score seleccionado: 0.947


,593
frequency_docs,1
sales_lines,1
units_sold,1.0
monetary_gross,309.5
margin_gross,0.0
avg_discount_pct,0.0
pct_discounted_lines,0.0
avg_unit_price,309.5
n_categories,0
n_products,1


,feature,local_weighted_signal,abs_value
0,gross_margin_rate,0.037371,0.037371
1,margin_gross,0.032193,0.032193
2,ciudad_Córdoba,0.029915,0.029915
3,recency_days,0.025564,0.025564
4,avg_order_value,0.025056,0.025056
5,provincia_Córdoba,0.022182,0.022182
6,monetary_gross,0.021772,0.021772
7,avg_discount_pct,0.021545,0.021545
8,pct_discounted_lines,0.020585,0.020585
9,n_categories,0.019932,0.019932


## 4. Tool 2: RAG sobre reglas CRM, consentimiento y campañas


In [5]:
rag_query = "fidelización CRM consentimiento marketing presión comercial campaña reactivación cupón cliente Platino"
append_trace(trace, "tool_call", "rag.retrieve", {"query": rag_query, "top_k": 5})
evidence = retrieve(rag_index, rag_query, top_k=5)
append_trace(trace, "evidence", "documentos recuperados", evidence[["file_name", "chunk_id", "score"]].head(3).to_dict(orient="records"))
display(evidence[["rank", "score", "file_name", "chunk_id", "area"]])


,rank,score,file_name,chunk_id,area
26,1,0.163863,FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx,FS-KB-07-CH000,05_CRM
27,2,0.110820,FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx,FS-KB-07-CH001,05_CRM
4,3,0.044778,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH004,01_Politicas
3,4,0.041280,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH003,01_Politicas
23,5,0.040277,FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.docx,FS-KB-06-CH000,04_Catalogo


## 5. Tool 3: CRM y guardrails


In [6]:
customer_id = row_raw.iloc[0].get("customer_id_std")
crm_row = crm[crm["customer_id_std"] == customer_id].head(1)
append_trace(trace, "tool_call", "crm.lookup", {"customer_id": customer_id})

if len(crm_row):
    consent = str(crm_row.iloc[0].get("consentimiento_marketing", "")).lower()
    status = crm_row.iloc[0].get("estado_cliente", "")
    tier = crm_row.iloc[0].get("tier_fidelizacion", "")
else:
    consent, status, tier = "", "", ""

allowed_consent = consent in ["1", "sí", "si", "yes", "true", "s"]
not_blocked = "bloqueado" not in str(status).lower()
guardrail_ok = allowed_consent and not_blocked

append_trace(trace, "tool_result", "crm y consentimiento", {"customer_id": customer_id, "tier": tier, "estado": status, "consentimiento": consent})
append_trace(trace, "guardrail", "verificación de consentimiento y estado de cliente", {"approved_for_activation": guardrail_ok})

display(crm_row.T if len(crm_row) else pd.DataFrame({"warning": ["cliente no encontrado en CRM"]}))
print("Guardrail activación aprobado:", guardrail_ok)


,warning
0,cliente no encontrado en CRM


Guardrail activación aprobado: False


## 6. Respuesta final del agente


In [7]:
recommendation = recommendation_from_score(score, customer_context, evidence)
if not guardrail_ok:
    final_answer = (
        "No se recomienda activar automáticamente la campaña para este cliente. "
        "Aunque el score de inactividad es alto, la comprobación de consentimiento/estado no permite una acción directa. "
        "Se puede registrar el caso para revisión o buscar otro cliente elegible."
    )
else:
    final_answer = recommendation["recommendation"]

append_trace(trace, "final_answer", "respuesta al usuario", {"answer": final_answer})
print(final_answer)


No se recomienda activar automáticamente la campaña para este cliente. Aunque el score de inactividad es alto, la comprobación de consentimiento/estado no permite una acción directa. Se puede registrar el caso para revisión o buscar otro cliente elegible.


## 7. Evaluar la traza del agente

            Evaluamos si la interacción contiene las capas mínimas: intención, plan, tool calls, evidencia, guardrails y respuesta final. Esta evaluación no mide si el texto suena bien; mide si el agente es auditable.


In [8]:
trace_df = pd.DataFrame(trace)
display(trace_df)
display(evaluate_agent_trace(trace))

trace_path = ARTIFACT_DIR / "demo_03_agent_trace.json"
trace_path.write_text(json.dumps(trace, ensure_ascii=False, indent=2), encoding="utf-8")
print("Traza guardada:", trace_path)


,step,type,detail,payload
0,0,user_request,"Selecciona clientes Platino con alto riesgo de inactividad y prepara una propuesta de campaña respetando consentimiento, margen y presión comercial.",NaN
1,1,intent,reactivación comercial de clientes de alto valor con restricciones de gobierno,None
2,2,plan,1) obtener cliente con score alto; 2) explicar drivers; 3) consultar reglas CRM; 4) comprobar consentimiento; 5) generar recomendación con cautelas,None
3,3,tool_call,churn_model.predict_proba + explicación local,{'target': 'label_inactive_90d'}
4,4,tool_result,cliente prioritario y señales principales,"{'score': 0.9469, 'signals': ['gross_margin_rate', 'margin_gross', 'ciudad_Córdoba', 'recency_days', 'avg_order_value']}"
5,5,tool_call,rag.retrieve,"{'query': 'fidelización CRM consentimiento marketing presión comercial campaña reactivación cupón cliente Platino', 'top_k': 5}"
6,6,evidence,documentos recuperados,"[{'file_name': 'FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx', 'chunk_id': 'FS-KB-07-CH000', 'score': 0.1638628451698096}, {'file_name': 'FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx',..."
7,7,tool_call,crm.lookup,{'customer_id': None}
8,8,tool_result,crm y consentimiento,"{'customer_id': None, 'tier': '', 'estado': '', 'consentimiento': ''}"
9,9,guardrail,verificación de consentimiento y estado de cliente,{'approved_for_activation': False}


,capa,cumple,comentario
0,comprensión,True,traza presente
1,planificación,True,traza presente
2,uso de herramientas,True,traza presente
3,evidencia,True,traza presente
4,seguridad,True,traza presente
5,respuesta final,True,traza presente


Traza guardada: C:\repos\frasohome-demos\escenarios\09-genai-interpretabilidad\artifacts\demo_03_agent_trace.json


## 8. Lectura para el instructor

            El punto clave es separar tres niveles: el agente puede seleccionar bien las herramientas, pero aun así tener prohibida la activación por consentimiento; puede tener una respuesta razonable, pero sin evidencia; o puede tener buena evidencia, pero un plan incorrecto. Evaluar agentes implica revisar la cadena completa.
